In [1]:
import torch
import pandas as pd
from torch import nn
from datasets import load_dataset
from tqdm.notebook import tqdm

In [2]:
device = torch.device('mps')

In [3]:
ds = load_dataset("ashraq/movielens_ratings")

In [139]:
ds_train = pd.DataFrame(ds['validation'])[['movie_id', 'user_id', 'rating']]
ds_test = pd.DataFrame(ds['validation'])[['movie_id', 'user_id', 'rating']]

In [140]:
n_users = ds_train['user_id'].max()+1
n_movies = ds_train['movie_id'].max()+1

In [141]:
n_users, n_movies

(np.int64(44087), np.int64(15582))

In [142]:
class MovieDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.movies = torch.tensor(df['movie_id'].values, dtype=torch.long).to(device)
        self.users = torch.tensor(df['user_id'].values, dtype=torch.long).to(device)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.movies)

    def __getitem__(self, idx):
        return torch.stack([self.movies[idx], self.users[idx]]), self.ratings[idx]

In [143]:
dataset = MovieDataset(ds_train)
validation_dataset = MovieDataset(ds_test)

In [144]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=True)
valid_dataloader = torch.utils.data.DataLoader(validation_dataset, batch_size=256, shuffle=True)

In [170]:
class PMF(nn.Module):
    def __init__(self, n_movies, n_users, embed_size, rate_range):
        super().__init__()
        #self.movie = nn.Parameter(torch.randn(n_movies, embed_size, dtype=torch.float32)*0.01)
        #self.movie_bias = nn.Parameter(torch.zeros(n_movies, dtype=torch.float32))
        #self.user = nn.Parameter(torch.randn(n_users, embed_size, dtype=torch.float32)*0.01)
        #self.user_bias = nn.Parameter(torch.zeros(n_users, dtype=torch.float32))
        self.movie = nn.Embedding(n_movies, embed_size)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.user = nn.Embedding(n_users, embed_size)
        self.user_bias = nn.Embedding(n_users, 1)
        
        # Initialize weights the same way we did manually
        self.movie.weight.data.normal_(0, 0.1)
        self.user.weight.data.normal_(0, 0.1)
        #self.movie_bias.weight.data.zeros()
        #self.user_bias.weight.data.zeros()
        
        self.rate_range = rate_range

    # x.shape ([batch_size, ])
    def forward(self, x):
        movies = self.movie(x[:,0])
        users = self.user(x[:,1])
        return torch.sigmoid((movies*users).sum(1)+self.movie_bias(x[:,0])+self.user_bias(x[:,1]))*self.rate_range

In [171]:
model = PMF(n_movies, n_users, 100, 5.5)

In [172]:
model.to('mps')

PMF(
  (movie): Embedding(15582, 100)
  (movie_bias): Embedding(15582, 1)
  (user): Embedding(44087, 100)
  (user_bias): Embedding(44087, 1)
)

In [173]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [174]:
loss_fn = nn.MSELoss()

In [175]:
epochs = 100

In [176]:
for i in tqdm(range(epochs)):
    for input, target in tqdm(dataloader):
        optimizer.zero_grad()
        output = model(input.to(device))
        loss = loss_fn(output, target.to(device))
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        loss = 0
        c = 0
        for valid, target in valid_dataloader:
            output = model(valid.to(device))
            loss += loss_fn(output, target.to(device))
            c += 1
        loss /= c
        print("epoch", i, "average validation loss", loss)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/387 [00:00<?, ?it/s]

/Users/matheoledevehat/fastai-course/.venv/lib/python3.11/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([256])) that is different to the input size (torch.Size([256, 256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/matheoledevehat/fastai-course/.venv/lib/python3.11/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([227])) that is different to the input size (torch.Size([227, 227])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch 0 average validation loss tensor(3.3578, device='mps:0')


  0%|          | 0/387 [00:00<?, ?it/s]

...

epoch 19 average validation loss tensor(2.2328, device='mps:0')


  0%|          | 0/387 [00:00<?, ?it/s]

epoch 20 average validation loss tensor(2.2120, device='mps:0')


  0%|          | 0/387 [00:00<?, ?it/s]

KeyboardInterrupt: 